In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib.pyplot as plt


In [31]:
df_races = pd.read_csv("csv/races.csv")
df_laps = pd.read_csv("csv/laps.csv") 

In [32]:
hilfe_map = {
    1: "No Assistance",
    2: "With Driving Coach",
    3: "With Ideal Driving Line"
}

df_races["help_label"] = df_races["help"].map(hilfe_map)
df_laps["help_label"] = df_laps["help"].map(hilfe_map)


In [33]:
grouped = (
    df_races
    .groupby(["help", "racetrack"])["total_duration"]
    .mean()
    .reset_index()
)

grouped

,help,racetrack,total_duration
0,1.0,1.0,296.488750
1,1.0,2.0,379.925000
2,1.0,3.0,349.718571
3,2.0,1.0,263.850000
4,2.0,2.0,420.692500
5,2.0,3.0,391.995000
6,3.0,1.0,284.466667
7,3.0,2.0,367.702857
8,3.0,3.0,406.722500


In [34]:
mean_lap = (
    df_laps
    .groupby(["help_label", "lap"])["duration"]
    .mean()
    .reset_index()
)

plt.figure()

for hilfe in mean_lap["help_label"].unique():
    subset = mean_lap[mean_lap["help_label"] == hilfe]
    plt.plot(subset["lap"], subset["duration"], marker="o", label=hilfe)

plt.xlabel("Lap number")
plt.ylabel("Average lap duration (s)")
plt.title("Lap Duration by Assistance System")
plt.legend()
plt.tight_layout()
plt.savefig("pictures/lap_improvement_by_help_condition.png")
plt.close()

In [35]:
df_races_encoded = pd.get_dummies(
    df_races,
    columns=["racetrack", "help", "experience", "race_number"],
    drop_first=True
)

df_laps_encoded = pd.get_dummies(
    df_laps,
    columns=["racetrack", "lap", "help", "experience", "race_number"],
    drop_first=True
)

df_races_encoded.columns = [
    "name",
    "id",
    "total_duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]


df_laps_encoded.columns = [
    "name",
    "id",
    "duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "lap_2",
    "lap_3",
    "lap_4",
    "lap_5",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]

df_races_encoded.to_csv("csv/races_one_hot.csv", index=False)
df_laps_encoded.to_csv("csv/laps_one_hot.csv", index=False) 

In [36]:
X = df_races_encoded[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "experience_2",
            "experience_3",
            "race_number_2",
            "race_number_3",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.737
Model:                            OLS   Adj. R-squared:                  0.698
Method:                 Least Squares   F-statistic:                     18.94
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           3.59e-13
Time:                        14:50:06   Log-Likelihood:                -327.28
No. Observations:                  63   AIC:                             672.6
Df Residuals:                      54   BIC:                             691.9
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           349.1991     17.569     19.876

In [37]:

lap_cols = [c for c in df_laps_encoded.columns if c.startswith("lap_")]
lap_cols = sorted(lap_cols, key=lambda s: int(s.split("_")[1]))

def _decode_lap(row):
    for col in lap_cols:
        if row[col] == 1:
            return int(col.split("_")[1])
    return 1

df_laps_encoded["lap"] = df_laps_encoded.apply(_decode_lap, axis=1)

df_laps_encoded.to_csv("csv/laps_one_hot_decoded.csv", index=False)

In [38]:
X = df_laps_encoded[
        [
            "help_2",
            "help_3",
            "lap",
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "experience_2",
            "experience_3",
            "race_number_2",
            "race_number_3",
        ]
    ]
y = df_laps_encoded["duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

formula = "duration ~ racetrack_2 + racetrack_3 + help_2 + help_3 + experience_2 + experience_3 + race_number_2 + race_number_3 + help_2 + help_3 + lap + lap*help_2 + lap*help_3"
            
model = sm.OLS.from_formula(formula, data=df_laps_encoded).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               duration   R-squared:                       0.650
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                     51.23
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           1.44e-62
Time:                        14:50:06   Log-Likelihood:                -1199.8
No. Observations:                 315   AIC:                             2424.
Df Residuals:                     303   BIC:                             2469.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                74.83